# ES4304 Tutorial 1 — Assignment: California Eaton Fire (January 2025)

This notebook processes Sentinel-2 L1C imagery of the Eaton Fire (California) and produces NDVI and NBR images across multiple dates to show the progression and recovery of the wildfire.

| | |
|---|---|
| **Location** | Altadena / Pasadena, Los Angeles County (~34.20°N, 118.07°W) |
| **Fire start** | 7 January 2025 |
| **100% contained** | 31 January 2025 |
| **Area burned** | 14,021 acres (~57 km²) |
| **Sensor** | Sentinel-2 L1C (Top of Atmosphere reflectance) |
| **Radiometric baseline** | N0400+ — DN offset correction applied: `(DN − 1000) / 10000` |

### Outputs produced per scene

| File | Index | Bands | Resolution |
|---|---|---|---|
| `B02_B03_B04_B08_10m_{label}.tif` | — | Blue, Green, Red, NIR | 10 m |
| `B05_B06_B07_B8A_B11_B12_20m_{label}.tif` | — | Red Edge, NIR narrow, SWIR | 20 m |
| `NDVI_10m_{label}.tif` | NDVI | (B08 − B04) / (B08 + B04) | 10 m |
| `NBR_20m_{label}.tif` | NBR | (B8A − B12) / (B8A + B12) | 20 m |

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_origin
from rasterio.windows import from_bounds
from pyproj import Transformer

## Helper Functions

The three core functions below are identical to the Merapi tutorial notebook. The utility functions — `find_img_data`, `find_tile_date_prefix` and `read_metadata` — auto-detect the IMG_DATA path and extract ULX/ULY/CRS from the scene's `MTD_TL.xml`, so you only need to specify the `.SAFE` folder name in the configuration cell. One new utility function - `bbox_to_window` - crops the loaded data to the specified bounding box.

In [ ]:
def create_tif_variable_from_jp2(img_data_dir, *filenames, window=None):
    """
    Read one or more Sentinel-2 JP2 band files from img_data_dir
    and return a multiband int32 array, north-up.

    Parameters
    ----------
    img_data_dir : Path
        Path to the IMG_DATA folder (auto-detected by find_img_data).
    *filenames : str
        JP2 filenames, one per band, in the order to be stacked.
    window : rasterio.windows.Window, optional
        If provided, only the pixels within this window are read.
        Compute with bbox_to_window() before calling this function.
        If None, the full tile is read.

    Returns
    -------
    numpy.ndarray — (rows, cols, num_bands), int32, north-up.

    Note: rasterio reads JP2 files north-up natively, so flipud
    (used in the original MATLAB code) is not needed here.
    """
    img_data = []
    for filename in filenames:
        with rasterio.open(Path(img_data_dir) / filename) as src:
            img_data.append(src.read(1, window=window).astype(np.int32))
    return np.stack(img_data, axis=-1)

In [ ]:
def save_as_geotiff(composite_image, crs_code, output_filename,
                    transform=None, ulx=None, uly=None, xdim=None, ydim=None):
    """
    Save a composite image array as a georeferenced GeoTIFF file.

    Parameters
    ----------
    composite_image : numpy.ndarray
        2-D (rows, cols) or 3-D (rows, cols, num_bands).
    crs_code : int
        EPSG code.
    output_filename : str
        Filename without extension; .tif is appended automatically.
    transform : affine.Affine, optional
        Affine transform for the output raster. When a bounding box is
        used, pass src.window_transform(window) here so the GeoTIFF is
        correctly registered to the cropped area, not the full tile origin.
        If None, transform is computed from ulx/uly/xdim/ydim.
    ulx, uly : float, optional
        Upper-left corner coordinates. Used only when transform is None.
    xdim : float, optional
        Pixel width (positive, e.g. 10). Used only when transform is None.
    ydim : float, optional
        Pixel height (negative north-up, e.g. -10). Used only when
        transform is None.
    """
    if composite_image.ndim == 2:
        composite_image = composite_image[:, :, np.newaxis]

    rows, cols, num_bands = composite_image.shape

    if transform is None:
        if ulx is None:
            raise ValueError(
                "Provide either 'transform' or all of 'ulx', 'uly', 'xdim', 'ydim'."
            )
        transform = from_origin(ulx, uly, xdim, abs(ydim))

    crs = CRS.from_epsg(crs_code)
    output_path = str(output_filename) + '.tif'

    with rasterio.open(
        output_path, 'w', driver='GTiff',
        height=rows, width=cols, count=num_bands,
        dtype=composite_image.dtype, crs=crs, transform=transform,
    ) as dst:
        for i in range(num_bands):
            dst.write(composite_image[:, :, i], i + 1)

    print(f'  Saved: {output_path}')

In [ ]:
def bbox_to_window(src, lon_min, lat_min, lon_max, lat_max):
    """
    Convert a lat/lon bounding box to a rasterio Window
    in the CRS of the opened dataset.
    """
    # Transform lat/lon to the raster's CRS (e.g. UTM)
    transformer = Transformer.from_crs(
        CRS.from_epsg(4326),   # WGS84 lat/lon
        src.crs,               # e.g. EPSG:32611
        always_xy=True,        # lon/lat input order
    )
    x_min, y_min = transformer.transform(lon_min, lat_min)
    x_max, y_max = transformer.transform(lon_max, lat_max)

    return from_bounds(x_min, y_min, x_max, y_max, src.transform)

### Auto-Detection Utilities

These functions inspect the downloaded `.SAFE` folder so you do not need to manually locate or copy sub-directory names or metadata values.

| Function | What it finds |
|---|---|
| `find_img_data(safe_dir)` | `GRANULE/*/IMG_DATA` path |
| `find_tile_date_prefix(img_data_dir)` | Filename prefix, e.g. `T11SLT_20250102T182731` |
| `read_metadata(safe_dir)` | ULX, ULY, and EPSG code from `MTD_TL.xml` |

In [ ]:
def find_img_data(safe_dir):
    """
    Locate the IMG_DATA folder inside a Sentinel-2 .SAFE directory.

    Parameters
    ----------
    safe_dir : str or Path
        Path to the .SAFE folder.

    Returns
    -------
    Path to the IMG_DATA folder.
    """
    matches = list(Path(safe_dir).glob('GRANULE/*/IMG_DATA'))
    if not matches:
        raise FileNotFoundError(
            f'No IMG_DATA folder found in {safe_dir}.\n'
            f'Check that the .SAFE folder is correctly unzipped.'
        )
    return matches[0]


def find_tile_date_prefix(img_data_dir):
    """
    Derive the tile+date prefix shared by all JP2 band filenames.

    For example, given 'T11SLT_20250102T182731_B02.jp2' in IMG_DATA,
    this returns 'T11SLT_20250102T182731'.

    Parameters
    ----------
    img_data_dir : Path
        Path to the IMG_DATA folder.

    Returns
    -------
    str — the tile+date prefix.
    """
    matches = list(Path(img_data_dir).glob('*_B02.jp2'))
    if not matches:
        raise FileNotFoundError(
            f'No *_B02.jp2 file found in {img_data_dir}.\n'
            f'Check that the .SAFE folder is correctly unzipped.'
        )
    return matches[0].stem.replace('_B02', '')


def read_metadata(safe_dir):
    """
    Extract ULX, ULY, and EPSG code from MTD_TL.xml inside a .SAFE folder.

    Parameters
    ----------
    safe_dir : str or Path
        Path to the .SAFE folder.

    Returns
    -------
    tuple of (ulx: float, uly: float, epsg: int)
    """
    matches = list(Path(safe_dir).glob('GRANULE/*/MTD_TL.xml'))
    if not matches:
        raise FileNotFoundError(f'No MTD_TL.xml found in {safe_dir}.')

    root = ET.parse(matches[0]).getroot()

    # {*} matches any XML namespace prefix (Python 3.8+)
    ulx  = float(root.findall('.//{*}ULX')[0].text)
    uly  = float(root.findall('.//{*}ULY')[0].text)
    epsg = int(root.findall('.//{*}HORIZONTAL_CS_CODE')[0].text.split(':')[-1])

    return ulx, uly, epsg

## Configuration

1. Unzip all downloaded `.SAFE` folders into the `data/` sub-folder.
   *On Windows*, keep the repository somewhere short — `C:\ES4304\` rather
   than a deep folder under Documents or OneDrive. `.SAFE` folders nest deeply
   enough that the full path to a band file can pass the old 260-character
   limit, and the unzip then fails part-way with "path too long".
2. Replace each `CHANGE_ME` with the exact `.SAFE` folder name and a human-readable date label.
3. Run all cells top to bottom — everything else is auto-detected.

**Suggested dates** (check the Copernicus browser for cloud-free scenes):

| Scene | Suggested date | Rationale |
|---|---|---|
| Pre-fire | ~1-6 January 2025 | Baseline healthy vegetation |
| Co-fire | ~10-18 January 2025 | Active burn scar visible |
| Post-fire | ~16-28 January 2025 | Fire extinguished |
| Recovery | June-July 2025 | Clear vegetation regrowth |

In [ ]:
# ── STUDENT TEMPLATE ──────────────────────────────────────────
# Fill in your own .SAFE folder names, labels, and dates below.

# Root directory containing all unzipped .SAFE folders
data_dir = Path.cwd() / 'data'

# Define your scenes — replace CHANGE_ME with actual values
scenes = [
    {
        'safe_folder': 'CHANGE_ME.SAFE',
        'label':       'pre_fire',
        'date':        'CHANGE_ME',         
    },
    {
        'safe_folder': 'CHANGE_ME.SAFE',
        'label':       'co_fire',
        'date':        'CHANGE_ME',         
    },
    {
        'safe_folder': 'CHANGE_ME.SAFE',
        'label':       'post_fire',
        'date':        'CHANGE_ME',         
    },
    {
        'safe_folder': 'CHANGE_ME.SAFE',
        'label':       'recovery',
        'date':        'CHANGE_ME',         
    },
]

# Bounding box for the fire area (lat/lon, WGS84).
# Set to None to load the full tile (slower, uses more memory).
LON_MIN, LAT_MIN = None, None
LON_MAX, LAT_MAX = None, None

# Path to the output folder. Do not change.
Path("outputs").mkdir(exist_ok=True)
print(f"Output   : {Path('outputs').resolve()}")

## Processing

The loop below processes all scenes in `data/` automatically:

1. Locates `IMG_DATA` and reads metadata from `MTD_TL.xml`
2. Loads 10 m and 20 m band groups
3. Saves band composite GeoTIFFs
4. Computes and saves NDVI (10 m) with DN offset correction
5. Computes and saves NBR (20 m) with DN offset correction

Output filenames include the scene label so each run does not overwrite the others.

**NDVI** uses 10 m bands (B08, B04):
$$NDVI = \frac{B08 - B04}{B08 + B04}$$

**NBR** uses 20 m bands (B8A, B12):
$$NBR = \frac{B8A - B12}{B8A + B12}$$

| Index | Healthy vegetation | Burned / bare |
|---|---|---|
| NDVI | High positive (0.6–0.9) | Near zero or negative |
| NBR | High positive | Strongly negative |

**Band indices in arrays (0-based Python):**

`tif_10m`: B02(0) B03(1) **B04(2)** **B08(3)**

`tif_20m`: B05(0) B06(1) B07(2) **B8A(3)** B11(4) **B12(5)**

In [ ]:
results = []   # stores arrays for the visualisation cell

for scene in scenes:
    safe_dir = data_dir / scene['safe_folder']
    label    = scene['label']
    date     = scene['date']

    print(f"\n{'='*55}")
    print(f"Scene : {date}  ({label})")
    print(f"{'='*55}")

    # ── Auto-detect paths and metadata ──────────────────────────
    img_data_dir   = find_img_data(safe_dir)
    prefix         = find_tile_date_prefix(img_data_dir)
    ulx, uly, epsg = read_metadata(safe_dir)

    print(f"  IMG_DATA : {img_data_dir.relative_to(data_dir)}")
    print(f"  Prefix   : {prefix}")
    print(f"  ULX/ULY  : {ulx}, {uly}  (EPSG:{epsg})")

    # ── Compute windowed reads and transforms ────────────────────
    # When a bounding box is defined, only the cropped area is read
    # from disk — much faster and more memory-efficient than loading
    # the full 110×110 km tile.
    #
    # IMPORTANT: the windowed transform (src.window_transform) is
    # passed to save_as_geotiff so the GeoTIFF is registered to the
    # cropped area. Using the original ulx/uly would misregister the
    # output to the full-tile origin, placing it in the wrong location
    # in QGIS.
    if LON_MIN is not None:
        with rasterio.open(img_data_dir / f'{prefix}_B02.jp2') as src:
            window_10m    = bbox_to_window(src, LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
            transform_10m = src.window_transform(window_10m)
        with rasterio.open(img_data_dir / f'{prefix}_B8A.jp2') as src:
            window_20m    = bbox_to_window(src, LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
            transform_20m = src.window_transform(window_20m)
        print(f"  Bounding box: LON [{LON_MIN}, {LON_MAX}]  LAT [{LAT_MIN}, {LAT_MAX}]")
    else:
        window_10m, window_20m = None, None
        transform_10m = from_origin(ulx, uly, 10, 10)
        transform_20m = from_origin(ulx, uly, 20, 20)
        print("  No bounding box — loading full tile.")

    # ── 10 m bands: B02, B03, B04, B08 ─────────────────────────
    print("\n  Loading 10 m bands...")
    tif_10m = create_tif_variable_from_jp2(
        img_data_dir,
        f'{prefix}_B02.jp2',   # Blue   (490 nm)
        f'{prefix}_B03.jp2',   # Green  (560 nm)
        f'{prefix}_B04.jp2',   # Red    (665 nm)
        f'{prefix}_B08.jp2',   # NIR    (842 nm)
        window=window_10m,
    )
    print(f"  tif_10m  : {tif_10m.shape}  {tif_10m.dtype}")
    save_as_geotiff(tif_10m, epsg, f'outputs/B02_B03_B04_B08_10m_{label}',
                    transform=transform_10m)

    # ── 20 m bands: B05, B06, B07, B8A, B11, B12 ───────────────
    print("  Loading 20 m bands...")
    tif_20m = create_tif_variable_from_jp2(
        img_data_dir,
        f'{prefix}_B05.jp2',   # Red Edge 1 (705 nm)
        f'{prefix}_B06.jp2',   # Red Edge 2 (740 nm)
        f'{prefix}_B07.jp2',   # Red Edge 3 (783 nm)
        f'{prefix}_B8A.jp2',   # NIR narrow (865 nm)  ← NBR NIR
        f'{prefix}_B11.jp2',   # SWIR-1    (1610 nm)
        f'{prefix}_B12.jp2',   # SWIR-2    (2190 nm)  ← NBR SWIR
        window=window_20m,
    )
    print(f"  tif_20m  : {tif_20m.shape}  {tif_20m.dtype}")
    save_as_geotiff(tif_20m, epsg, f'outputs/B05_B06_B07_B8A_B11_B12_20m_{label}',
                    transform=transform_20m)

    # ── NDVI ────────────────────────────────────────────────────
    # Formula : (B08 - B04) / (B08 + B04)
    # Indices : tif_10m[:,:,3] = B08 (NIR),  tif_10m[:,:,2] = B04 (Red)
    print("  Computing NDVI...")
    nir = (tif_10m[:, :, 3].astype(np.float32) - 1000) / 10000.0
    red = (tif_10m[:, :, 2].astype(np.float32) - 1000) / 10000.0
    with np.errstate(invalid='ignore'):
        denom_ndvi = nir + red
        ndvi = np.where(denom_ndvi != 0, (nir - red) / denom_ndvi, 0.0)
    print(f"  NDVI     : min={ndvi.min():.3f}  max={ndvi.max():.3f}")
    save_as_geotiff(ndvi, epsg, f'outputs/NDVI_10m_{label}', transform=transform_10m)

    # ── NBR ─────────────────────────────────────────────────────
    # Formula : (B8A - B12) / (B8A + B12)
    # Indices : tif_20m[:,:,3] = B8A (NIR narrow), tif_20m[:,:,5] = B12 (SWIR-2)
    # B8A (20 m) is used instead of B8 (10 m) to match B12 resolution.
    print("  Computing NBR...")
    nir_b8a  = (tif_20m[:, :, 3].astype(np.float32) - 1000) / 10000.0
    swir_b12 = (tif_20m[:, :, 5].astype(np.float32) - 1000) / 10000.0
    with np.errstate(invalid='ignore'):
        denom_nbr = nir_b8a + swir_b12
        nbr = np.where(denom_nbr != 0, (nir_b8a - swir_b12) / denom_nbr, 0.0)
    print(f"  NBR      : min={nbr.min():.3f}  max={nbr.max():.3f}")
    save_as_geotiff(nbr, epsg, f'outputs/NBR_20m_{label}', transform=transform_20m)

    # ── Store arrays for visualisation ───────────────────────────
    # With a bounding box the cropped arrays are small enough to keep
    # in memory directly. Without a bbox, subsample to avoid storing
    # multiple full 10980×10980 tiles simultaneously.
    if LON_MIN is not None:
        results.append({
            'label': label,
            'date':  date,
            'ndvi':  ndvi,
            'nbr':   nbr,
        })
    else:
        results.append({
            'label': label,
            'date':  date,
            'ndvi':  ndvi[::10, ::10],   # ~100 m display resolution
            'nbr':   nbr[::5,  ::5],     # ~100 m display resolution
        })
    print("  Done.")

print(f"\nAll {len(results)} scenes processed.")

## dNBR — Differenced Normalized Burn Ratio

dNBR compares a pre-fire NBR scene against a post-fire NBR scene to directly
map burn severity across the fire perimeter.

$$dNBR = NBR_{pre} - NBR_{post}$$

A healthy pre-fire pixel has high NBR; the same pixel after burning has low or
negative NBR. The difference is therefore large and positive in severely burned
areas, and near zero in unburned areas.

| dNBR value | Burn severity |
|---|---|
| > 0.66 | High severity |
| 0.44 – 0.66 | Moderate-high severity |
| 0.27 – 0.44 | Moderate-low severity |
| 0.10 – 0.27 | Low severity |
| −0.10 – 0.10 | Unburned |
| < −0.10 | Enhanced regrowth (post-fire vegetation flush) |

*Thresholds from the USGS burn severity classification (Key & Benson, 2006).*

**Why dNBR rather than NBR alone?**  
A single NBR scene can be ambiguous — low NBR values could indicate burned
vegetation, dry vegetation, bare soil, or water. dNBR removes the pre-existing
landscape variation, isolating the *change* caused by the fire. The burn
perimeter and internal severity gradients become much clearer.

**Scenes used:** pre-fire NBR (`NBR_20m_pre_fire.tif`) minus co-fire NBR
(`NBR_20m_co_fire.tif`), both saved by the processing loop above.

**Tutorial Step:** Load `dNBR_20m.tif` into QGIS. Apply a **RdYlGn_r**
(reversed) colour ramp — red for high burn severity, green for unburned.
Use the USGS thresholds above as break points in the symbology.

In [ ]:
# ── dNBR — Differenced Normalized Burn Ratio ─────────────────────
# dNBR = NBR_pre - NBR_post
# Higher positive values indicate greater burn severity.
# Uses the pre-fire and co-fire NBR GeoTIFFs saved above.

with rasterio.open('outputs/NBR_20m_pre_fire.tif') as src:
    nbr_pre       = src.read(1).astype(np.float32)
    transform_nbr = src.transform
    epsg_nbr      = src.crs.to_epsg()

with rasterio.open('outputs/NBR_20m_co_fire.tif') as src:
    nbr_post = src.read(1).astype(np.float32)

dNBR = nbr_pre - nbr_post

print(f"dNBR  min: {dNBR.min():.3f}  max: {dNBR.max():.3f}")
save_as_geotiff(dNBR, epsg_nbr, 'outputs/dNBR_20m',
                transform=transform_nbr)

## Visualisation

The grid below shows the NDVI and NBR for all scenes side by side, allowing direct comparison of the fire progression.

- **Row 1 — NDVI:** Vegetation health. Low values (red) indicate vegetation loss; recovering areas return to green over time.
- **Row 2 — NBR:** Burn severity. Burned areas appear strongly negative (red); recovering areas return toward positive values.

> For a detailed map layout with basemap, scale bar, and legend, load the output `.tif` files into QGIS and apply a **RdYlGn** colour ramp with Min = −1, Max = +1.

In [ ]:
n_scenes = len(results)
fig, axes = plt.subplots(2, n_scenes, figsize=(5 * n_scenes, 3 * n_scenes),
                         constrained_layout=True)

for col, result in enumerate(results):

    # ── NDVI (top row) ──────────────────────────────────────────
    ax_ndvi = axes[0, col]
    im_ndvi = ax_ndvi.imshow(result['ndvi'], cmap='RdYlGn', vmin=-1, vmax=1)
    ax_ndvi.set_title(result['date'], fontsize=11)
    ax_ndvi.set_xlabel('~10 m pixels')
    if col == 0:
        ax_ndvi.set_ylabel('NDVI', fontsize=12, fontweight='bold')
    else:
        ax_ndvi.set_yticks([])

    # ── NBR (bottom row) ────────────────────────────────────────
    ax_nbr = axes[1, col]
    im_nbr = ax_nbr.imshow(result['nbr'], cmap='RdYlGn', vmin=-1, vmax=1)
    ax_nbr.set_xlabel('~20 m pixels')
    if col == 0:
        ax_nbr.set_ylabel('NBR', fontsize=12, fontweight='bold')
    else:
        ax_nbr.set_yticks([])

# Shared colourbars spanning the full row
fig.colorbar(im_ndvi, ax=axes[0, :], location='right', shrink=0.6,
             fraction=0.02, pad=0.01, label='NDVI')
fig.colorbar(im_nbr,  ax=axes[1, :], location='right', shrink=0.6,
             fraction=0.02, pad=0.01, label='NBR')

plt.suptitle('Eaton Fire — NDVI and NBR Progression', fontsize=14,
             fontweight='bold', y=1.01)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dNBR[::5, ::5], cmap='RdYlGn_r', vmin=-1, vmax=1)
ax.set_title('dNBR (Pre-fire − Co-fire)', fontsize=12)
ax.set_xlabel('~100 m pixels')
plt.colorbar(im, ax=ax, label='dNBR')
plt.tight_layout()
plt.show()